# 📊 Day 3, Topics 5 & 6 — Data Type Conversion & String Cleaning

Getting dtypes right is essential before any analysis. Wrong types cause silent errors — e.g., averaging strings instead of numbers.


In [1]:
#Day 3, Topic 5: Data Type Conversion
import pandas as pd
df = pd.read_csv('titanic.csv')

## Topic 5 — Data Type Conversion

### Why dtype matters
- `int64` vs `object`: can't do `df['Age'].mean()` if Age is stored as strings
- `category` dtype: saves memory and speeds up `.groupby()` on low-cardinality columns
- `bool` vs `int64`: makes logic explicit, enables `.sum()` = count of True
- `datetime64`: unlocks `.dt.year`, `.dt.month`, time arithmetic

### Conversion tools

| Tool | Use |
|------|-----|
| `.astype(type)` | Direct conversion when data is already clean |
| `pd.to_numeric(series, errors='coerce')` | Convert strings to numbers; set non-numeric to NaN |
| `pd.to_datetime(series, errors='coerce')` | Convert strings to datetime; set invalid to NaT |
| `.str.replace()` + `.astype()` | Clean messy strings (e.g., `'$1,200'`) then convert |


### Current Dtypes — Always Check First
`df.dtypes` shows what Pandas inferred. Common issues:
- Numbers stored as `object` (happened during CSV reading with mixed values like `'N/A'`)
- Categorical columns stored as `int64` (like Pclass: 1, 2, 3)
- Boolean columns stored as `int64` (like Survived: 0, 1)


In [2]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

### `.astype()` — Direct Conversion
Works only when the column is already in a convertible state (no strings, no NaN for int).
```python
df['Survived'].astype(bool)       # 0 → False, 1 → True
df['Pclass'].astype('category')   # int → category (saves memory, semantic clarity)
df['ID'].astype(str)              # int → string
```


In [3]:
#.astype() – Direct Type Conversion
df['Survived'] = df['Survived'].astype(bool)
df['Survived'].dtype

dtype('bool')

### `pd.to_numeric()` — Safe String-to-Number Conversion
`errors='coerce'` converts non-numeric values to `NaN` instead of raising an error.  
```python
pd.to_numeric(['10', '20', 'N/A', '30'], errors='coerce')
# → [10.0, 20.0, NaN, 30.0]
```
`errors='ignore'` leaves problematic values unchanged (returns object dtype).  
`errors='raise'` (default) raises a `ValueError` on any non-numeric value.


In [5]:
#Converting Strings to Numbers – pd.to_numeric()
prices = pd.Series(['10', '20', 'N/A', '30'])
numeric_prices = pd.to_numeric(prices, errors='coerce')
numeric_prices

0    10.0
1    20.0
2     NaN
3    30.0
dtype: float64

### `pd.to_datetime()` — Safe String-to-Datetime Conversion
`errors='coerce'` converts invalid date strings to `NaT` (Not a Time — datetime's NaN equivalent).  
After conversion you get access to the `.dt` accessor:
```python
df['Date'].dt.year    # extract year
df['Date'].dt.month   # extract month
df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
```


In [6]:
#Converting to Datetime – pd.to_datetime()
dates = pd.Series(['2024-01-01', '2024-01-02', 'Invalid'])
datetime_dates = pd.to_datetime(dates, errors='coerce')
datetime_dates


0   2024-01-01
1   2024-01-02
2          NaT
dtype: datetime64[ns]

### Cleaning Messy String Numbers
Real data often has currency symbols, commas, or percent signs:
```python
'$1,200.50' → strip '$' and ',' → '1200.50' → float(1200.50)
```
Steps:
1. `.str.replace('$', '', regex=False)` — remove dollar sign
2. `.str.replace(',', '', regex=False)` — remove commas
3. `.astype(float)` or `pd.to_numeric(...)` — convert


In [9]:
# Create a messy 'Fare_str' column with dollar signs and commas
df['Fare_str'] = '$' + df['Fare'].astype(str)

# Convert it back to float
df['Fare_clear'] = df['Fare_str'].str.replace('$', '').astype(float)


---
### 📝 Practice — Data Type Conversion


In [10]:
#Practice Activity: Data Type Conversion
import pandas as pd

df = pd.read_csv('titanic.csv')

#### Task 1 — Survived to Bool
`df['Survived'].astype(bool)` — makes the column semantically clear. `True` = survived.


In [12]:
#Task 1: Convert the Survived column from int64 to bool. Verify with .dtypes.

df['Survived'] = df['Survived'].astype(bool)

df.dtypes

PassengerId      int64
Survived          bool
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

#### Task 2 — Pclass to Category
`df['Pclass'].astype('category')` — 1, 2, 3 become an ordered categorical.  
Memory usage drops significantly for large datasets with few unique values.


In [14]:
#Task 2: Convert the Pclass column to category type (it only takes values 1, 2, 3). Verify the new dtype.

df['Pclass'] = df['Pclass'].astype('category')
df['Pclass'].dtype

CategoricalDtype(categories=[1, 2, 3], ordered=False, categories_dtype=int64)

#### Task 3 — Clean Currency String then Convert
Create `Fare_str` with `'$'` prefix, then strip it and convert back to float.  
`.str.replace('$', '', regex=False)` then `pd.to_numeric(...)`.


In [18]:
"""Task 3: Create a new column Fare_str by adding a dollar sign and commas: '$' + df['Fare'].round(2).astype(str). 
Then use pd.to_numeric() with errors='coerce' to convert it back to float. Why does it fail? 
Then fix it by first stripping the dollar sign with .str.replace()."""

df['Fare_str'] = '$' + df['Fare'].round(2).astype(str)
#numeric_fare = pd.to_numeric(df['Fare_str'], errors='coerce')
df['Fare_clean'] = df['Fare_str'].str.replace('$', '').astype(float)

#### Task 4 — Simulate Datetime with `errors='coerce'`
`pd.to_datetime(['2024-01-01', '2024-02-01', 'Not a date'], errors='coerce')` → last entry becomes `NaT`.


In [19]:
"""Task 4: Simulate a date column: create a Series of strings ['2024-01-01', '2024-02-01', 'Not a date']. 
Use pd.to_datetime() with errors='coerce' to convert it. What value replaces the invalid entry?"""

dates = pd.Series(['2024-01-01', '2024-02-01', 'Not a date'])
valid_dates = pd.to_datetime(dates, errors='coerce')
valid_dates

0   2024-01-01
1   2024-02-01
2          NaT
dtype: datetime64[ns]

#### Task 5 (Bonus) — Extract Deck from Cabin
`df['Cabin'].str[0]` extracts the first character (e.g., `'C85'` → `'C'`).  
Converts Cabin from a sparse, messy string to a useful categorical deck indicator.


In [21]:
"""Task 5 (Bonus): The Cabin column contains values like 'C85'. Extract the letter part (the deck) 
into a new column Deck using .str[0]. Then convert Deck to category type."""

df['Deck'] = df['Cabin'].str[0]

df['Deck'] = df['Deck'].astype('category')

In [22]:
#Day 3, Topic 6: String Cleaning

import pandas as pd

df = pd.read_csv('titanic.csv')

## Topic 6 — String Cleaning

### The `.str` accessor — your vectorized string toolkit
```python
series.str.strip()              # remove leading/trailing whitespace
series.str.lower() / .upper()   # case normalization
series.str.replace(old, new)    # simple or regex replacement
series.str.extract(r'(regex)')  # pull captured groups into new columns
series.str.split('delim', expand=True)  # split → DataFrame of parts
series.str.len()                # string length per element
series.str[0]                   # character at position 0
```
All methods return `NaN` for missing values — no special handling needed.

### String cleaning workflow
1. `.strip()` — remove whitespace
2. `.lower()` — normalize case
3. `.replace()` — fix known bad values
4. `.extract()` / `.split()` — parse structure


### Basic Cleaning: Strip + Lower + Replace
`df['Name'].str.strip().str.lower().str.replace('.', '', regex=False)` — chain multiple operations.  
Each `.str` method returns a Series, so you can chain as many as needed.


In [26]:
# Strip whitespace (not really needed for Titanic, but good practice)
df['Name'] = df['Name'].str.strip()

# Convert all names to lowercase
df['Name_Lower'] = df['Name'].str.lower()

# Replace 'Miss' with 'Ms'
df['Name_replaced'] = df['Name'].replace('Miss', 'Ms')

# Extract last name (everything before the comma)
df['Last_Name'] = df['Name'].str.split(',').str[0]

### Chaining String Methods
```python
df['Name'].str.strip().str.lower().str.replace('.', '', regex=False)
```
Reads left-to-right like a pipeline: strip whitespace → lowercase → remove periods.  
This is more readable and efficient than applying each transformation separately.


In [27]:
#Chaining String Methods
# Clean and standardize: strip, lower, replace
df['Name_clean'] = df['Name'].str.strip().str.lower().str.replace('.', '')

### Handling NaN Before String Operations
Some `.str` methods fail on NaN. Safe options:
1. `df['Cabin'].fillna('')` — replace NaN with empty string before operating
2. Use `na=False` parameter where available (e.g., `.str.contains('X', na=False)`)


In [29]:
#Handling Missing Values with .str
df['Cabin'] = df['Cabin'].fillna('')
df['Cabin_clean'] = df['Cabin'].str.strip()

### Regex Replacement with `.str.replace()`
`df['Name'].str.replace(r'[^\w\s]', '', regex=True)` — remove all non-word, non-space characters (punctuation).  
The `regex=True` flag enables full Python `re` syntax.  
Common patterns: `r'[^a-zA-Z]'` (keep letters only), `r'\s+'` (collapse multiple spaces).


In [31]:
#Using Regular Expressions with .str.replace()
# Remove all punctuation from names

df['Name_no_punct'] = df['Name'].str.replace(r'[^\w\s]', '', regex=True)

---
### 📝 Practice — String Cleaning


In [32]:
#Practice Activity: String Cleaning

import pandas as pd

df = pd.read_csv('titanic.csv')

#### Task 1 — Strip + Lowercase
`df['Name'].str.strip().str.lower()` → creates `Name_clean`.  
Standardizes case so 'Mr.' and 'mr.' aren't treated as different values.


In [33]:
"""Task 1: Create a new column Name_clean that contains the Name column with leading and trailing whitespace 
removed, and converted to lowercase. Display the first 5 rows of Name and Name_clean."""

df['Name_clean'] = df['Name'].str.strip().str.lower()

df[['Name', 'Name_clean']].head()

,Name,Name_clean
0,"Braund, Mr. Owen Harris","braund, mr. owen harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...","cumings, mrs. john bradley (florence briggs th..."
2,"Heikkinen, Miss. Laina","heikkinen, miss. laina"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)","futrelle, mrs. jacques heath (lily may peel)"
4,"Allen, Mr. William Henry","allen, mr. william henry"


#### Task 2 — Remove Periods
`.str.replace('.', '', regex=False)` — removes all periods from names.  
`regex=False` is important here: without it, `.` is a regex wildcard matching any character!


In [34]:
"""Task 2: The Name column contains titles like 'Mr.', 'Mrs.', 'Miss.'. Use .str.replace() to replace all periods (.) 
with an empty string in the Name column. Create a new column Name_no_periods. Display the first 5 rows."""

df['Name_no_periods'] = df['Name'].str.replace('.', '')
df[['Name', 'Name_no_periods']].head()

,Name,Name_no_periods
0,"Braund, Mr. Owen Harris","Braund, Mr Owen Harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...","Cumings, Mrs John Bradley (Florence Briggs Tha..."
2,"Heikkinen, Miss. Laina","Heikkinen, Miss Laina"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)","Futrelle, Mrs Jacques Heath (Lily May Peel)"
4,"Allen, Mr. William Henry","Allen, Mr William Henry"


#### Task 3 — Extract First Name
`df['Name'].str.split(',')` → split at comma → first name is in `str[1]` after further split.  
Name format: `'LastName, Title. FirstName MiddleName'`  
Chain: `str.split(',').str[1]` → `str.split('.').str[1]` → `.str.strip()`


In [48]:
"""Task 3: Extract the first name from the Name column. The format is "LastName, Title. FirstName" 
(sometimes with additional names). Use .str.split(',') to get the part after the comma, 
then .str.split('.') to get the part after the period, and finally .str.strip() to clean whitespace. 
Create a new column FirstName. Display Name and FirstName for the first 5 rows."""


# Extract the part after the title and take the first word
df['First_Name'] = df['Name'].str.extract(r'[A-Za-z]+\.\s+(\w+)')

#### Task 4 — Extract Title with Regex
`df['Name'].str.extract(r'([A-Za-z]+)\.')` captures the word immediately before a period.  
This is the standard Titanic title extraction — produces 'Mr', 'Mrs', 'Miss', 'Master', 'Dr', etc.


In [46]:
"""Task 4: Create a new column Title by extracting the title (e.g., 'Mr', 'Mrs', 'Miss') 
from Name. Use .str.extract(r'([A-Za-z]+)\.') and then convert to lowercase. Display the value counts of Title.
"""

df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.', expand=False).str.lower()
df['Title'].value_counts()

Title
mr          517
miss        182
mrs         125
master       40
dr            7
rev           6
mlle          2
major         2
col           2
countess      1
capt          1
ms            1
sir           1
lady          1
mme           1
don           1
jonkheer      1
Name: count, dtype: int64

#### Task 5 (Bonus) — Regex Cleaning
Use `.str.replace(r'[^\w\s]', '', regex=True)` to strip all punctuation from names.  
Then `.str.strip()` to clean up any trailing spaces left behind.


In [ ]:
"""Task 5 (Bonus): Some passengers have names in quotes (e.g., "Braund, Mr. Owen Harris"—actually no quotes in Titanic, 
but imagine). Use .str.replace() with a regex to remove any double quotes " from the Name column"""

df['Name_no_quotes'] = df['Name'].str.replace('"', '', regex=True)

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888               Johnston, Miss. Catherine Helen Carrie
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: object